# SHAP pipeline

For each tract, get feature contributions to **P_CANCER** and **P_RESP**.
Uses `../.data/analysis_minimal.csv` (run **build_analysis_minimal** notebook first).

**Outputs:** `../.data/tract_shap.csv`, `../.data/shap_importance.csv`

In [2]:
%pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.0/556.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 5.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 5.7 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split

data_dir = Path("../.data")
minimal_csv = data_dir / "analysis_minimal.csv"

feature_columns = [
    "P_PM25", "P_OZONE", "P_DSLPM", "P_PTRAF",
    "P_LDPNT", "P_PNPL", "P_PRMP", "P_PTSDF", "P_UST", "P_PWDIS",
    "DEMOGIDX_2",
]

In [4]:
if not minimal_csv.exists():
    raise FileNotFoundError(f"Run build_analysis_minimal notebook first to create {minimal_csv}")

df = pd.read_csv(minimal_csv)
available = [c for c in feature_columns if c in df.columns]
if len(available) != len(feature_columns):
    raise ValueError(f"Missing features: {set(feature_columns) - set(available)}")

X = df[available]
id_cols = df[["ID", "ST_ABBREV", "CNTY_NAME"]].copy()
print(f"Loaded {len(df):,} tracts")

Loaded 81,001 tracts


In [5]:
# Subsample for faster SHAP (set to None to use full data)
max_rows = 100_000
if len(df) > max_rows:
    idx, _ = train_test_split(df.index, train_size=max_rows, random_state=42, stratify=df["ST_ABBREV"])
    df = df.loc[idx].reset_index(drop=True)
    X = df[available]
    id_cols = df[["ID", "ST_ABBREV", "CNTY_NAME"]].copy()
    print(f"Using subsample of {len(df):,} tracts (max_rows={max_rows})")

## Train models and compute SHAP per outcome

In [6]:
results = []

for outcome_name, outcome_col in [("cancer", "P_CANCER"), ("resp", "P_RESP")]:
    if outcome_col not in df.columns:
        continue
    y = df[outcome_col]
    print(f"--- {outcome_col} ---")
    model = GradientBoostingRegressor(n_estimators=100, max_depth=4, min_samples_leaf=20, random_state=42)
    model.fit(X, y)
    print(f"  R² (train): {model.score(X, y):.4f}")
    explainer = shap.TreeExplainer(model, X)
    shap_vals = explainer.shap_values(X)
    for i, feat in enumerate(available):
        results.append({"outcome": outcome_name, "feature": feat, "mean_abs_shap": float(pd.Series(shap_vals[:, i]).abs().mean())})
        id_cols[f"shap_{outcome_name}_{feat}"] = shap_vals[:, i]
    print("  SHAP done.")

--- P_CANCER ---
  R² (train): 0.4425


 99%|===================| 80470/81001 [01:22<00:00]        

  SHAP done.
--- P_RESP ---
  R² (train): 0.5584


 99%|===================| 80085/81001 [01:20<00:00]        

  SHAP done.


In [7]:
out = id_cols.copy()
shap_cols = [c for c in out.columns if c.startswith("shap_")]

# Top-3 driver columns per outcome
for outcome_name in ["cancer", "resp"]:
    cols = [c for c in shap_cols if c.startswith(f"shap_{outcome_name}_")]
    if not cols:
        continue
    feat_order = [c.replace(f"shap_{outcome_name}_", "") for c in cols]
    vals = out[cols].values
    abs_vals = np.abs(vals)
    rank_order = abs_vals.argsort(axis=1)[:, ::-1]
    for r in range(3):
        out[f"top_{outcome_name}_{r+1}_feature"] = [feat_order[rank_order[i, r]] for i in range(len(out))]
        out[f"top_{outcome_name}_{r+1}_shap"] = [vals[i, rank_order[i, r]] for i in range(len(out))]

In [8]:
importance_df = pd.DataFrame(results)
importance_wide = importance_df.pivot(index="feature", columns="outcome", values="mean_abs_shap").reset_index()
importance_wide.to_csv(data_dir / "shap_importance.csv", index=False)
print("Saved", data_dir / "shap_importance.csv")

out.to_csv(data_dir / "tract_shap.csv", index=False)
print(f"Saved tract-level SHAP -> {data_dir / 'tract_shap.csv'} ({len(out):,} rows)")

Saved ../.data/shap_importance.csv
Saved tract-level SHAP -> ../.data/tract_shap.csv (81,001 rows)


In [9]:
importance_wide

outcome,feature,cancer,resp
0,DEMOGIDX_2,1.588232,1.713754
1,P_DSLPM,5.945696,10.339100
2,P_LDPNT,0.698532,0.270444
3,P_OZONE,2.970687,1.385659
4,P_PM25,6.351262,6.966046
5,P_PNPL,0.843042,0.972604
6,P_PRMP,0.253902,0.367750
7,P_PTRAF,0.743404,0.662302
8,P_PTSDF,1.037669,0.514266
9,P_PWDIS,1.392599,0.904711


In [10]:
out.head(10)

,ID,ST_ABBREV,CNTY_NAME,shap_cancer_P_PM25,shap_cancer_P_OZONE,shap_cancer_P_DSLPM,shap_cancer_P_PTRAF,shap_cancer_P_LDPNT,shap_cancer_P_PNPL,shap_cancer_P_PRMP,...,top_cancer_2_feature,top_cancer_2_shap,top_cancer_3_feature,top_cancer_3_shap,top_resp_1_feature,top_resp_1_shap,top_resp_2_feature,top_resp_2_shap,top_resp_3_feature,top_resp_3_shap
0,1001020100,AL,Autauga,4.637315,1.162094,-0.711722,0.984673,0.470398,-0.035771,0.125842,...,DEMOGIDX_2,-1.996023,P_PTSDF,-1.884605,P_PM25,6.049634,P_DSLPM,-3.932190,DEMOGIDX_2,-1.845281
1,1001020200,AL,Autauga,5.484013,1.780774,1.308917,-0.791565,0.494384,-0.135185,0.222374,...,DEMOGIDX_2,2.036245,P_PTSDF,-1.919662,P_PM25,6.912185,DEMOGIDX_2,3.644793,P_DSLPM,-1.297205
2,1001020300,AL,Autauga,7.302305,1.775882,2.929437,-0.292083,0.977579,-0.067297,0.090473,...,P_DSLPM,2.929437,P_OZONE,1.775882,P_PM25,6.527113,P_DSLPM,2.204089,DEMOGIDX_2,-0.774797
3,1001020400,AL,Autauga,5.556396,1.892516,3.902308,-0.950253,0.477207,-0.072882,0.135291,...,P_DSLPM,3.902308,P_PWDIS,-2.519178,P_DSLPM,7.499510,P_PM25,6.776983,DEMOGIDX_2,-1.802127
4,1001020501,AL,Autauga,6.285008,0.888528,4.148046,-0.635551,-3.827794,-0.093906,0.165149,...,P_DSLPM,4.148046,P_LDPNT,-3.827794,P_DSLPM,8.343560,P_PM25,7.026655,DEMOGIDX_2,-1.465535
5,1001020502,AL,Autauga,6.164865,0.903326,4.343088,-0.948077,-3.749783,-0.006376,0.186964,...,P_DSLPM,4.343088,P_LDPNT,-3.749783,P_DSLPM,8.484168,P_PM25,7.105181,DEMOGIDX_2,-1.617963
6,1001020503,AL,Autauga,6.496254,-0.329726,4.411739,-0.339946,-1.099735,-0.087181,0.061626,...,P_DSLPM,4.411739,P_PWDIS,1.632168,P_DSLPM,8.301045,P_PM25,7.045179,DEMOGIDX_2,-1.637439
7,1001020600,AL,Autauga,5.110885,1.276492,1.699969,-0.259448,-0.434740,-0.027078,0.142910,...,P_DSLPM,1.699969,DEMOGIDX_2,1.692845,P_PM25,6.408880,P_DSLPM,-1.220204,P_PWDIS,0.813312
8,1001020700,AL,Autauga,7.724473,2.726005,3.285697,-0.766019,1.065125,-0.106086,0.106766,...,P_DSLPM,3.285697,P_OZONE,2.726005,P_PM25,7.703953,P_DSLPM,4.127014,DEMOGIDX_2,3.628791
9,1001020801,AL,Autauga,3.584931,0.580224,-6.327350,0.921325,0.129770,0.121140,-0.141250,...,P_PM25,3.584931,DEMOGIDX_2,-3.417774,P_DSLPM,-10.546741,P_PM25,5.324340,DEMOGIDX_2,-3.807805
